# Chapter 9 §9.7: Financial Analyst with RLM (Recursive Language Model)

This notebook accompanies Chapter 9 §9.7 of *Context Engineering with DSPy*: **Financial Analyst with RLM (Recursive Language Model)**.

**Required environment variables (set in your `.env`):**

- `OPENAI_API_KEY`

**Two execution modes:**

- **RLM mode (default — no extra setup):** Uses `dspy.RLM`, which runs tool-call loops with sub-LM delegation. Works out of the box with just `OPENAI_API_KEY`.
- **ProgramOfThought (PoT) mode:** If you want to swap `dspy.RLM` for `dspy.ProgramOfThought`, you'll need [Deno](https://deno.land/) installed locally (PoT uses Pyodide via Deno to execute generated Python code safely). RLM is strictly better for this use case — Deno is only required if you want to compare against the PoT baseline.

## Overview

```
9.7 Financial Analyst with RLM (Recursive Language Model)

Demonstrates the most advanced context engineering pattern in the book:
1. Document parsing as Context Editing — structure raw financial data
2. RLM for financial analysis — the model decides what context it needs
3. Trajectory evaluation — catch hallucinated numbers and lost focus

Three failure modes make financial analysis especially hard:
- Context Overflow: a 10-K filing is 80,000+ tokens. Add earnings transcripts
  and quarterly data — you can't fit it all in one context window.
- Context Poisoning: a hallucinated revenue figure in step 3 cascades into
  wrong growth rates, wrong valuations, wrong conclusions.
- Context Drift: a 15-step analysis that starts asking about profitability
  but ends up exploring market cap after the model loses the thread.

RLM solves all three. Instead of the engineer deciding what to parse and
extract up front, the RLM writes Python code to explore documents, calls
sub-LMs for semantic queries, and iteratively builds only the context it needs.

Failure mode: Context Overflow + Context Poisoning + Context Drift
Technique: Context Editing + Tool Loadout + RLM
Agentic pattern: Orchestrator-Workers + Autonomous Agents
Key DSPy feature: dspy.RLM (replaces ProgramOfThought — strictly better)

Usage:
    python financial_analyst.py                                    # full demo
    python financial_analyst.py --ticker AAPL --question "P/E ratio vs industry"
    python financial_analyst.py --compare AAPL MSFT GOOGL          # multi-stock
    python financial_analyst.py --skip-trajectory                   # skip eval demo
```


In [ ]:
%pip install -r ../requirements.txt -q

## Imports and LM configuration

In [ ]:
import os
import json
from dotenv import load_dotenv

import dspy
from dspy.evaluate import Evaluate

load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
lm = dspy.LM("openai/gpt-4o-mini", api_key=api_key)
dspy.configure(lm=lm)

## Simulated financial data (replace with yfinance, SEC EDGAR, etc.)

In [ ]:
STOCK_DATA = {
    "AAPL": {
        "ticker": "AAPL", "company": "Apple Inc.",
        "current_price": 178.25, "previous_close": 176.50,
        "52_week_high": 199.62, "52_week_low": 164.08,
        "volume": 52_000_000, "market_cap": 2_780_000_000_000,
        "sector": "Technology",
    },
    "MSFT": {
        "ticker": "MSFT", "company": "Microsoft Corporation",
        "current_price": 378.91, "previous_close": 375.23,
        "52_week_high": 384.30, "52_week_low": 309.45,
        "volume": 23_000_000, "market_cap": 2_810_000_000_000,
        "sector": "Technology",
    },
    "GOOGL": {
        "ticker": "GOOGL", "company": "Alphabet Inc.",
        "current_price": 142.18, "previous_close": 140.93,
        "52_week_high": 153.78, "52_week_low": 121.46,
        "volume": 18_500_000, "market_cap": 1_750_000_000_000,
        "sector": "Technology",
    },
}

FINANCIALS = {
    "AAPL": {
        "revenue": 383_285_000_000, "net_income": 96_995_000_000,
        "gross_margin": 0.461, "operating_margin": 0.302,
        "profit_margin": 0.253, "pe_ratio": 29.4,
        "eps": 6.06, "dividend_yield": 0.005,
        "debt_to_equity": 1.73, "current_ratio": 0.99,
        "free_cash_flow": 99_584_000_000,
        "revenue_growth_yoy": 0.028,
        "quarterly_revenue": [
            {"q": "Q1 2024", "revenue": 119_575_000_000, "net_income": 33_916_000_000},
            {"q": "Q4 2023", "revenue": 89_498_000_000, "net_income": 22_956_000_000},
            {"q": "Q3 2023", "revenue": 81_797_000_000, "net_income": 19_881_000_000},
            {"q": "Q2 2023", "revenue": 94_836_000_000, "net_income": 24_160_000_000},
        ],
    },
    "MSFT": {
        "revenue": 211_915_000_000, "net_income": 72_361_000_000,
        "gross_margin": 0.695, "operating_margin": 0.442,
        "profit_margin": 0.342, "pe_ratio": 35.2,
        "eps": 9.68, "dividend_yield": 0.007,
        "debt_to_equity": 0.45, "current_ratio": 1.77,
        "free_cash_flow": 59_475_000_000,
        "revenue_growth_yoy": 0.159,
        "quarterly_revenue": [
            {"q": "Q1 2024", "revenue": 61_858_000_000, "net_income": 21_870_000_000},
            {"q": "Q4 2023", "revenue": 56_517_000_000, "net_income": 20_081_000_000},
            {"q": "Q3 2023", "revenue": 52_857_000_000, "net_income": 18_299_000_000},
            {"q": "Q2 2023", "revenue": 53_036_000_000, "net_income": 18_884_000_000},
        ],
    },
    "GOOGL": {
        "revenue": 307_394_000_000, "net_income": 73_795_000_000,
        "gross_margin": 0.572, "operating_margin": 0.277,
        "profit_margin": 0.240, "pe_ratio": 25.8,
        "eps": 5.80, "dividend_yield": 0.0,
        "debt_to_equity": 0.09, "current_ratio": 2.10,
        "free_cash_flow": 69_495_000_000,
        "revenue_growth_yoy": 0.085,
        "quarterly_revenue": [
            {"q": "Q1 2024", "revenue": 80_539_000_000, "net_income": 23_662_000_000},
            {"q": "Q4 2023", "revenue": 86_310_000_000, "net_income": 20_687_000_000},
            {"q": "Q3 2023", "revenue": 76_693_000_000, "net_income": 19_689_000_000},
            {"q": "Q2 2023", "revenue": 74_604_000_000, "net_income": 18_368_000_000},
        ],
    },
}

# Simulated 10-K excerpt (long document for RLM to explore)
TENK_EXCERPTS = {
    "AAPL": """
APPLE INC. FORM 10-K (Excerpts)

ITEM 1. BUSINESS
Apple Inc. designs, manufactures, and markets smartphones, personal computers,
tablets, wearables, and accessories. The Company's products include iPhone, Mac,
iPad, and Wearables, Home and Accessories.

ITEM 6. SELECTED FINANCIAL DATA
(in millions, except per share amounts)

                    2024        2023        2022
Net sales          $383,285    $383,933    $394,328
Cost of sales      $206,312    $214,137    $223,546
Gross margin       $176,973    $169,796    $170,782
Operating income   $115,846    $114,301    $119,437
Net income          $96,995     $96,733    $105,820
EPS (diluted)        $6.06       $6.00       $6.14
Cash & equivalents  $29,965     $29,965     $23,646
Total debt         $108,040    $111,088    $132,480

ITEM 7. MANAGEMENT'S DISCUSSION AND ANALYSIS
Revenue decreased 0.2% in fiscal 2024 compared to fiscal 2023. iPhone revenue
increased 2% year-over-year. Services revenue grew 13% and reached an all-time
high of $85.2 billion. Greater China revenue declined 8%.

Products revenue decreased 3%, primarily driven by lower Mac and iPad sales.
Wearables, Home and Accessories revenue decreased 7%.

Gross margin percentage increased to 46.1% from 44.1%, driven by cost savings
and a shift toward higher-margin Services.

The Company returned over $100 billion to shareholders through dividends and
share repurchases during fiscal 2024.
""",
}

## Data retrieval tools (usable by RLM)

In [ ]:
def get_stock_price(ticker: str) -> str:
    """Get current stock price and market data for a ticker.

    Args:
        ticker: Stock ticker symbol (e.g., 'AAPL', 'MSFT', 'GOOGL')

    Returns:
        JSON string with current price, volume, and market cap
    """
    data = STOCK_DATA.get(ticker.upper())
    if not data:
        return json.dumps({"error": f"Ticker {ticker} not found. Available: {list(STOCK_DATA.keys())}"})
    return json.dumps(data, indent=2)

def get_financials(ticker: str) -> str:
    """Get company financial metrics: revenue, margins, ratios, quarterly data.

    Args:
        ticker: Stock ticker symbol (e.g., 'AAPL', 'MSFT', 'GOOGL')

    Returns:
        JSON string with comprehensive financial data
    """
    data = FINANCIALS.get(ticker.upper())
    if not data:
        return json.dumps({"error": f"Ticker {ticker} not found. Available: {list(FINANCIALS.keys())}"})
    return json.dumps(data, indent=2)

def get_10k_filing(ticker: str) -> str:
    """Get excerpts from the company's most recent 10-K filing.

    This returns a long document. Use programmatic exploration (string
    search, section extraction) rather than reading it all at once.

    Args:
        ticker: Stock ticker symbol

    Returns:
        10-K filing text (excerpt)
    """
    text = TENK_EXCERPTS.get(ticker.upper())
    if not text:
        return f"No 10-K filing available for {ticker}. Available: {list(TENK_EXCERPTS.keys())}"
    return text

## RLM-based financial analyst

In [ ]:
class FinancialAnalystRLM(dspy.Module):
    """Financial analyst using RLM for context-managed analysis.

    Unlike ProgramOfThought (which generates one code block and executes it),
    RLM iteratively explores data: it writes code, sees the output, writes
    more code, calls sub-LMs for semantic analysis, and builds up the answer.

    This is strictly better for financial analysis because:
    1. The model decides what data to explore (not the engineer)
    2. Sub-LM calls handle semantic queries ("What drove revenue growth?")
    3. Iterative exploration catches errors before they cascade
    """

    def __init__(self, sub_lm=None):
        super().__init__()
        tools = [get_stock_price, get_financials, get_10k_filing]

        self.analyst = dspy.RLM(
            "ticker, question -> analysis: str",
            max_iterations=10,
            max_llm_calls=5,
            tools=tools,
            sub_lm=sub_lm,
        )

    def forward(self, ticker, question):
        return self.analyst(ticker=ticker, question=question)

## Trajectory evaluation

In [ ]:
class EvaluateTrajectory(dspy.Signature):
    """Evaluate a financial analysis trajectory for quality issues.

    Check for:
    1. Context Poisoning: Are any numbers used in calculations that don't
       match the source data? Hallucinated numbers cascade into wrong conclusions.
    2. Context Drift: Did the analysis stay focused on the original question,
       or did it wander into unrelated territory?
    3. Completeness: Does the final answer actually address the question?

    Score each dimension independently. A correct answer reached through
    a flawed trajectory is still risky — it might not be correct next time.
    """

    question: str = dspy.InputField(desc="The original financial question")
    analysis: str = dspy.InputField(desc="The model's analysis/answer")
    source_data: str = dspy.InputField(desc="The raw financial data available")

    accuracy: float = dspy.OutputField(
        desc="1-10: Are the numbers in the analysis consistent with the source data?"
    )
    focus: float = dspy.OutputField(
        desc="1-10: Does the analysis stay focused on the original question?"
    )
    completeness: float = dspy.OutputField(
        desc="1-10: Does the analysis fully answer the question?"
    )
    issues: str = dspy.OutputField(
        desc="Any specific issues found (hallucinated numbers, drift, gaps)"
    )

def trajectory_metric(example, pred, trace=None):
    """Evaluate the quality of a financial analysis.

    Uses an LLM judge to check for poisoned numbers, drift, and completeness.
    This is more expensive than simple string matching but catches the
    failure modes that matter in financial analysis.
    """
    evaluator = dspy.Predict(EvaluateTrajectory)

    # Get the source data for verification
    ticker = example.ticker
    source = json.dumps({
        "stock": STOCK_DATA.get(ticker, {}),
        "financials": FINANCIALS.get(ticker, {}),
    }, indent=2)

    try:
        scores = evaluator(
            question=example.question,
            analysis=pred.analysis,
            source_data=source,
        )

        accuracy = float(scores.accuracy) / 10
        focus = float(scores.focus) / 10
        completeness = float(scores.completeness) / 10

        # Weight accuracy highest — wrong numbers are the most dangerous
        return 0.45 * accuracy + 0.30 * focus + 0.25 * completeness
    except (ValueError, TypeError):
        return 0.0

## Evaluation dataset

In [ ]:
QUESTIONS = [
    # Simple lookups (should be easy)
    {"ticker": "AAPL", "question": "What is Apple's current P/E ratio and how does it compare to its profit margin?"},
    {"ticker": "MSFT", "question": "Calculate Microsoft's year-over-year quarterly revenue growth from Q2 2023 to Q1 2024."},
    # Cross-company comparison (requires multiple data sources)
    {"ticker": "GOOGL", "question": "Compare Alphabet's debt-to-equity ratio and current ratio to Apple and Microsoft. Which company has the strongest balance sheet?"},
    # Deeper analysis (requires 10-K)
    {"ticker": "AAPL", "question": "Based on the 10-K filing, what segments drove Apple's revenue changes and what are the implications for future growth?"},
    # Calculation-heavy (where LLMs hallucinate most)
    {"ticker": "AAPL", "question": "Calculate Apple's free cash flow yield (FCF / market cap) and compare it to its dividend yield. Is the stock returning enough cash to shareholders?"},
    {"ticker": "MSFT", "question": "What is Microsoft's trailing twelve month revenue growth rate and how does it compare to its P/E ratio? Is the growth rate justifying the valuation?"},
]

def build_dataset():
    """Create evaluation dataset."""
    examples = [
        dspy.Example(
            ticker=q["ticker"],
            question=q["question"],
        ).with_inputs("ticker", "question")
        for q in QUESTIONS
    ]
    return examples[:4], examples[4:]

## Demos

In [ ]:
def demo_rlm_analysis():
    """Show RLM doing iterative financial analysis."""
    print("=" * 60)
    print("RLM FINANCIAL ANALYSIS")
    print("=" * 60)
    print()
    print("  RLM writes Python code to explore financial data iteratively.")
    print("  It calls sub-LMs for semantic analysis and builds answers")
    print("  step by step — the model decides what to explore, not us.")
    print()

    analyst = FinancialAnalystRLM()

    for q in QUESTIONS[:3]:
        result = analyst(ticker=q["ticker"], question=q["question"])

        print(f"\n--- {q['ticker']}: {q['question'][:60]}... ---")
        print(f"Analysis:\n  {result.analysis[:300]}...")

def demo_trajectory_eval():
    """Show trajectory evaluation catching quality issues."""
    print("\n" + "=" * 60)
    print("TRAJECTORY EVALUATION")
    print("=" * 60)
    print()
    print("  Trajectory evaluation catches three failure modes:")
    print("  1. Context Poisoning — hallucinated numbers in calculations")
    print("  2. Context Drift — analysis wanders from the question")
    print("  3. Incompleteness — question not fully answered")
    print()

    analyst = FinancialAnalystRLM()

    trainset, testset = build_dataset()
    evaluator = Evaluate(
        devset=trainset,
        metric=trajectory_metric,
        display_progress=False,
    )

    score = float(evaluator(analyst))
    print(f"  Trajectory quality score: {score:.1f}%")
    print(f"  Evaluated on {len(trainset)} financial questions")

    # Show detailed evaluation on one example
    print("\n  --- Detailed evaluation on one question ---")
    q = QUESTIONS[0]
    result = analyst(ticker=q["ticker"], question=q["question"])

    eval_sig = dspy.Predict(EvaluateTrajectory)
    source = json.dumps({
        "stock": STOCK_DATA.get(q["ticker"], {}),
        "financials": FINANCIALS.get(q["ticker"], {}),
    }, indent=2)

    try:
        scores = eval_sig(
            question=q["question"],
            analysis=result.analysis,
            source_data=source,
        )

        print(f"  Question: {q['question'][:60]}...")
        print(f"  Accuracy:     {scores.accuracy}/10")
        print(f"  Focus:        {scores.focus}/10")
        print(f"  Completeness: {scores.completeness}/10")
        print(f"  Issues:       {scores.issues[:150]}")
    except Exception as e:
        print(f"  Evaluation failed: {e}")

def demo_comparison():
    """Compare RLM vs. basic ChainOfThought on calculation-heavy questions."""
    print("\n" + "=" * 60)
    print("RLM vs. CHAIN OF THOUGHT")
    print("=" * 60)
    print()
    print("  ChainOfThought reasons in text — it does math in its head.")
    print("  RLM writes Python code — it does math with a calculator.")
    print("  For financial analysis, this difference matters.")
    print()

    # ChainOfThought baseline
    cot_analyst = dspy.ChainOfThought("ticker, question, stock_data, financials -> analysis: str")

    # RLM
    rlm_analyst = FinancialAnalystRLM()

    calc_questions = [q for q in QUESTIONS if "Calculate" in q["question"] or "calculate" in q["question"]]
    if not calc_questions:
        calc_questions = QUESTIONS[1:3]  # fallback to questions requiring computation

    for q in calc_questions:
        ticker = q["ticker"]
        source_stock = json.dumps(STOCK_DATA.get(ticker, {}))
        source_fin = json.dumps(FINANCIALS.get(ticker, {}))

        # CoT
        cot_result = cot_analyst(
            ticker=ticker,
            question=q["question"],
            stock_data=source_stock,
            financials=source_fin,
        )

        # RLM
        rlm_result = rlm_analyst(ticker=ticker, question=q["question"])

        # Evaluate both
        example = dspy.Example(ticker=ticker, question=q["question"]).with_inputs("ticker", "question")

        cot_pred = dspy.Prediction(analysis=cot_result.analysis)
        rlm_pred = dspy.Prediction(analysis=rlm_result.analysis)

        cot_score = trajectory_metric(example, cot_pred)
        rlm_score = trajectory_metric(example, rlm_pred)

        print(f"\n  Q: {q['question'][:60]}...")
        print(f"  CoT score: {cot_score:.3f}  |  RLM score: {rlm_score:.3f}")
        print(f"  CoT: {cot_result.analysis[:150]}...")
        print(f"  RLM: {rlm_result.analysis[:150]}...")

## Run the demo

The code below mirrors the `main()` block in the source script with the argparse CLI branches removed — it runs the full demo path end-to-end.

In [ ]:
# Full demo
demo_rlm_analysis()

if not False:
    demo_trajectory_eval()

demo_comparison()